# Pipeline NLP video pemrograman

Di Google Colab, cek **Runtime > Change runtime type > T4 GPU**, lalu jalankan sel 01. Sel 01 menampilkan GPU yang berhasil dialokasikan sebelum menyiapkan proyek. Saat pemilih file muncul, pilih **proyek-nlp-colab-v4.zip**. Jika paket sudah tersedia di `/content/proyek-nlp-colab-v4.zip`, sel langsung memakainya. Sel 01 mengekstrak proyek dan memasang dependensi. Untuk menjalankan lokal, buka notebook dari folder proyek.

Di Colab, pilih runtime GPU lalu jalankan sel secara berurutan. Latihan transkripsi berjalan di bagian 06 dan latihan peringkasan di bagian 13. Parameter latihan ada di bagian masing-masing. [Panduan proyek](README.md)

Untuk optimasi mBART IndoSum saja, jalankan bagian **01** lalu **13B**. Bagian 13B melatih kandidat LoRA secara terpisah di Colab.

## 01. Setup proyek dan lingkungan

In [ ]:
# Lokasi paket proyek di Colab
ZIP_PROYEK = "/content/proyek-nlp-colab-v4.zip"

import json
import os
import shutil
import subprocess
import sys
from datetime import datetime
from io import BytesIO
from pathlib import Path
from time import perf_counter
from zipfile import BadZipFile, ZipFile

from IPython.display import Markdown, display

try:
    from google import colab
except ImportError:
    colab = None
IS_COLAB = colab is not None
if IS_COLAB:
    nvidia_smi = shutil.which("nvidia-smi")
    gpu_probe = subprocess.run([nvidia_smi, "-L"], capture_output=True, text=True) if nvidia_smi else None
    if gpu_probe is None or gpu_probe.returncode or not any(
        line.startswith("GPU ") for line in gpu_probe.stdout.splitlines()
    ):
        raise RuntimeError(
            "Colab belum mengalokasikan GPU. Pilih Runtime > Change runtime type > T4 GPU, "
            "lalu sambungkan runtime dan jalankan ulang sel 01."
        )
    print(gpu_probe.stdout.strip(), flush=True)

    ROOT_COLAB = Path("/content/proyek-nlp")
    required_files = {
        "requirements-training.txt",
        "requirements-notebooks.txt",
        "notebooks/Pipeline_NLP_Video_Pemrograman.ipynb",
        "scripts/compare_partial_seq2seq.py",
        "scripts/optimize_mbart_indosum.py",
        "data/processed/indosum/train.csv",
        "data/processed/whisper_video/train.csv",
    }
    bundle_version = "colab-mbart-indosum-v4"
    version_file = "colab-bundle-version.txt"

    def bundle_problem(package):
        if version_file not in package.namelist():
            return "ZIP ini berasal dari paket lama dan belum memiliki penanda versi."
        version = package.read(version_file).decode("utf-8", errors="replace").strip()
        if version != bundle_version:
            return f"Versi paket {version!r}, sedangkan notebook memerlukan {bundle_version!r}."
        missing = sorted(required_files.difference(package.namelist()))
        if missing:
            return f"File wajib belum tersedia di ZIP: {', '.join(missing)}."
        return None

    marker = ROOT_COLAB / version_file
    if (not marker.is_file() or marker.read_text(encoding="utf-8").strip() != bundle_version
            or not all((ROOT_COLAB / name).is_file() for name in required_files)):
        archive = Path(ZIP_PROYEK).expanduser()
        extracted = False
        if archive.is_file():
            try:
                with ZipFile(archive) as package:
                    problem = bundle_problem(package)
                    if problem:
                        print(problem)
                    else:
                        package.extractall(ROOT_COLAB)
                        extracted = True
            except BadZipFile:
                print(f"File {archive.name} bukan ZIP yang utuh.")
        if not extracted:
            from google.colab import files

            print(f"Pilih {archive.name} dari komputer (sekitar 46 MB).")
            uploaded = files.upload()
            zip_files = [(name, content) for name, content in uploaded.items() if Path(name).suffix.lower() == ".zip"]
            if len(zip_files) != 1:
                raise ValueError(f"Pilih satu file {archive.name}, lalu jalankan kembali sel 01.")
            name, content = zip_files[0]
            try:
                with ZipFile(BytesIO(content)) as package:
                    if problem := bundle_problem(package):
                        raise ValueError(f"{name}: {problem} Gunakan {archive.name}.")
                    package.extractall(ROOT_COLAB)
            except BadZipFile as exc:
                raise ValueError(f"{name} bukan ZIP yang utuh. Unduh ulang {archive.name}.") from exc
            del uploaded, zip_files, content
    os.chdir(ROOT_COLAB)
    installed = ROOT_COLAB / ".colab-dependencies-installed"
    if not installed.exists():
        if not (shutil.which("ffprobe") and shutil.which("tesseract")):
            subprocess.run(["apt-get", "update", "-qq"], check=True)
            subprocess.run(["apt-get", "install", "-y", "-qq", "ffmpeg", "tesseract-ocr"], check=True)
        subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                        "-r", "requirements-training.txt", "-r", "requirements-notebooks.txt"], check=True)
        installed.touch()

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
from matplotlib.ticker import PercentFormatter

ROOT = next(path for path in (Path.cwd(), *Path.cwd().parents) if (path / "requirements-training.txt").is_file())
os.chdir(ROOT)
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

LOCAL_MODE = os.getenv("NLP_NOTEBOOK_MODE", "replay")
if LOCAL_MODE not in {"replay", "smoke", "pilot"}:
    raise ValueError("Gunakan replay, smoke, atau pilot.")
MBART_PYTHON = os.getenv("NLP_MBART_PYTHON", sys.executable)
if IS_COLAB:
    if not torch.cuda.is_available():
        raise RuntimeError("GPU terdeteksi, tetapi CUDA belum tersedia di PyTorch. Restart runtime GPU, lalu ulangi sel 01.")
    cuda_test = (torch.ones(1, device="cuda:0") + 1).item()
    if cuda_test != 2:
        raise RuntimeError("Tes operasi CUDA gagal. Sambungkan ulang runtime GPU dan jalankan kembali sel 01.")
    free_vram, total_vram = torch.cuda.mem_get_info(0)
    MBART_DEVICE = WHISPER_DEVICE = "cuda:0"
    print(f"CUDA aktif: {torch.cuda.get_device_name(0)} | "
          f"VRAM tersedia {free_vram / 1024**3:.1f} dari {total_vram / 1024**3:.1f} GiB")
    print(f"Tes operasi CUDA berhasil: {cuda_test:.0f}")
else:
    MBART_DEVICE = os.getenv("NLP_MBART_DEVICE", "cpu")
    WHISPER_DEVICE = os.getenv("NLP_WHISPER_DEVICE", "cpu")

def read_json(path):
    return json.loads(Path(path).read_text(encoding="utf-8"))

file = ROOT / "notebooks/Pipeline_NLP_Video_Pemrograman.ipynb"
replay = read_json(file)["metadata"]["project_replay"]
RUN = ROOT / "outputs/notebooks/integrated" / datetime.now().strftime("%Y%m%d-%H%M%S-%f")
SEED = 42
SPLITS = ("train", "validation", "test")
PALETTE = ["#627088", "#2858b8", "#b56d32"]
BEFORE = "Sebelum pelatihan tambahan"
AFTER = "Sesudah pelatihan tambahan"
plt.rcParams.update({"axes.spines.top": False, "axes.spines.right": False, "figure.dpi": 115})

def execute(module, *args, python=sys.executable):
    command = [str(python), "-u", "-m", module, *map(str, args)]
    with subprocess.Popen(command, cwd=ROOT, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                          text=True, encoding="utf-8", errors="replace",
                          env={**os.environ, "PYTHONIOENCODING": "utf-8"}) as process:
        try:
            for line in process.stdout:
                print(line, end="", flush=True)
        except KeyboardInterrupt:
            process.terminate()
            raise
        if process.wait():
            raise subprocess.CalledProcessError(process.returncode, command)

print(f"Proyek siap | perangkat transkripsi {WHISPER_DEVICE} | perangkat peringkasan {MBART_DEVICE}")

- **Colab** menjalankan latihan di bagian 06 dan 13 secara bawaan. Log menampilkan perangkat, jumlah contoh, progres latihan, dan hasil validasi.
- **Lokal** memakai laporan tersimpan. Mode `smoke` atau `pilot` dapat dipilih lewat `NLP_NOTEBOOK_MODE`.
- Hasil tersimpan di `outputs/notebooks/integrated/`. Salin folder hasil ke Drive sebelum runtime Colab berakhir.

## 02. Sumber data dan pembagian eksperimen

In [ ]:
BASE = ROOT / "data/processed"
lumina_master = BASE / "lumina/test.csv"
if lumina_master.is_file() and any(not (BASE / "lumina_split" / f"{split}.csv").is_file() for split in SPLITS):
    from scripts.prepare_lumina_splits import split_lumina

    split_lumina(lumina_master, BASE / "lumina_split")
sources = {
    "IndoSum": ("indosum", ("train", "validation", "test")),
    "Coding Video Mix": ("coding_video_mix", ("train", "validation", "test")),
    "NSS-ID": ("nss-id", ("train", "val", "test")),
    "Audio coding": ("whisper_video", SPLITS),
    "LUMINA": ("lumina_split", SPLITS),
}
tables = {
    name: {role: pd.read_csv(path, dtype={"id": str}, keep_default_na=False)
           for role, file_split in zip(SPLITS, splits, strict=True)
           if (path := BASE / folder / f"{file_split}.csv").is_file()}
    for name, (folder, splits) in sources.items()
}
holdout = json.loads((ROOT / "samples/notebook_video_holdout.json").read_text(encoding="utf-8"))
holdout_ids = {video["id"] for video in holdout}
for name in ("Coding Video Mix", "Audio coding"):
    known = set().union(*(set(df["video_id"]) - {""} for df in tables[name].values()))
    if known & holdout_ids:
        raise ValueError(f"Video akhir bertumpang tindih dengan {name}.")
group_keys = {"IndoSum": "id", "Coding Video Mix": "video_id", "NSS-ID": "speaker_id",
              "Audio coding": "video_id", "LUMINA": "speaker_id"}
for name, column in group_keys.items():
    seen = set()
    for frame in tables[name].values():
        values = set(frame[column]) - {""}
        if seen & values:
            raise ValueError(f"{column} berulang antarsplit pada {name}.")
        seen |= values
counts = pd.DataFrame({name: {split: len(df) for split, df in frames.items()}
                       for name, frames in tables.items()}).T.reindex(columns=SPLITS)
dataset_info = {
    "IndoSum": ("Peringkasan teks", "Artikel dan ringkasan Indonesia"),
    "Coding Video Mix": ("Peringkasan teks", "Teks tutorial pemrograman dan ringkasan"),
    "NSS-ID": ("Transkripsi audio", "NSS-ID Imperative, audio dan transkrip"),
    "Audio coding": ("Transkripsi audio", "Klip tutorial Python dan subtitle otomatis"),
    "LUMINA": ("Transkripsi audio", "LUMINA v4, audio dari klip video dan transkrip"),
}
catalog = pd.DataFrame.from_dict(dataset_info, orient="index", columns=["Bagian", "Isi dataset"])
display(Markdown("### Dataset dan jumlah contoh"))
dataset_table = catalog.join(counts.astype("Int64")).rename_axis("Dataset").reset_index()
dataset_table = dataset_table.rename(columns={"train": "Latih", "validation": "Validasi", "test": "Uji"})
display(dataset_table.style.hide(axis="index").format(escape="html", na_rep="—")
        .set_properties(subset=["Dataset", "Bagian", "Isi dataset"], **{"text-align": "left"}))

register = read_json(ROOT / "samples/summary_dataset_sources.json")
display(Markdown("**Sumber penyusun Coding Video Mix**"))
components = pd.DataFrame([{
    "Sumber": row["dataset"].removeprefix("Coding Video Mix ? "),
    "Pemakaian": row["role"], "Rujukan": row["url"],
} for row in register if row["local"] == "data/processed/coding_video_mix"])
display(components.style.hide(axis="index").format(escape="html")
        .set_properties(**{"text-align": "left", "overflow-wrap": "anywhere"}))
display(Markdown("**Video utuh untuk pengembangan dan uji eksploratif**"))
display(pd.DataFrame(holdout)[["title", "role", "video_url"]]
        .rename(columns={"title": "Judul", "role": "Peran", "video_url": "Tautan"})
        .style.hide(axis="index").format(escape="html").set_properties(**{"text-align": "left"}))
axis = np.log10(counts.fillna(0) + 1).plot.bar(rot=12, color=PALETTE, figsize=(9, 3.3))
axis.set(title="Contoh lokal per split", ylabel="log₁₀(1 + jumlah contoh)")
for bars, split in zip(axis.containers, SPLITS, strict=True):
    axis.bar_label(bars, labels=[str(int(n)) if pd.notna(n) else "" for n in counts[split]], padding=3)
axis.margins(y=0.18)
plt.show()

- Whisper adalah model transkripsi yang mengubah ucapan menjadi teks untuk peringkas. Subtitle yang cocok juga bisa menjadi sumber transkrip. Tiga peringkas menerima teks sumber dan belajar dari ringkasan acuan. CrossTask tersedia di split latih CSV Coding Video Mix. Pembanding tiga model memilih contoh berbahasa Indonesia.
- Jumlah di tabel menunjukkan isi CSV. Mode pelatihan mengambil subset. Laporan evaluasi mencatat jumlah contoh yang benar-benar digunakan.
- Evaluasi model memakai split uji dataset. Video utuh menguji seluruh pipeline. Break dipakai untuk pengembangan, sedangkan Format string dipakai untuk uji eksploratif. Untuk uji akhir independen, gunakan video baru setelah konfigurasi dikunci dan siapkan acuan manual bila ingin menghitung skor.

## 03. Eksplorasi audio

In [ ]:
from scripts.notebook_asr import SAMPLE_RATE, read_audio
from scripts.notebook_training import AUDIO

audio_records = []
if shutil.which("ffmpeg"):
    for dataset, (folder, _, media_key, group_key) in AUDIO.items():
        for split, frame in tables[dataset].items():
            sample = frame.sample(min(2, len(frame)), random_state=SEED)
            paths = [BASE / folder / item for item in sample[media_key]]
            times = [len(read_audio(path)) / SAMPLE_RATE for path in paths if path.is_file()]
            audio_records.append({"Dataset": dataset, "Split": split, "Klip": len(frame),
                                  "Grup unik": frame[group_key].nunique(),
                                  "Klip diukur": len(times), "Durasi contoh (detik)": np.median(times) if times else np.nan,
                                  "Median kata referensi": frame["transcript"].str.split().str.len().median()})
if not audio_records:
    audio_records = replay.get("notebook_eda", {}).get("audio", [])
if audio_records:
    audio_stats = pd.DataFrame(audio_records)
    display(audio_stats.round(2))
    if audio_stats["Durasi contoh (detik)"].notna().any():
        axis = audio_stats.pivot(index="Dataset", columns="Split", values="Durasi contoh (detik)").reindex(
            columns=SPLITS
        ).plot.bar(rot=0, color=PALETTE, figsize=(8, 2.8), title="Median durasi klip yang diukur")
        axis.set_ylabel("Detik")
        plt.show()
else:
    print("Manifest audio dan FFmpeg diperlukan untuk menghitung statistik durasi.")

- LUMINA dan NSS-ID mengukur ucapan Indonesia umum. Audio coding menguji istilah domain, dengan subtitle otomatis sebagai acuan yang dapat mengandung salah dengar.
- Durasi diukur pada maksimal dua klip acak per split. Median kata referensi dihitung dari seluruh split lokal.
- Setiap korpus menyumbang subset latih. Validasi memakai rata-rata WER per korpus agar korpus besar tidak mendominasi pemilihan epoch.

## 04. Eksplorasi pasangan teks dan strategi penggunaan data

In [ ]:
from src.seq2seq_models import load_tokenizer, model_files

try:
    mbart_path = model_files("mbart")
except OSError:
    mbart_path = None
tokenizer = load_tokenizer("mbart", mbart_path) if mbart_path else None
text_records = []
if tokenizer:
    for dataset in ("IndoSum", "Coding Video Mix"):
        for split, frame in tables[dataset].items():
            valid = frame[frame["text"].str.strip().ne("") & frame["summary"].str.strip().ne("")]
            if dataset == "Coding Video Mix":
                valid = valid[valid["source_lang"].eq("id_ID") & valid["target_lang"].eq("id_ID")]
            if valid.empty:
                continue
            sample = valid.sample(min(100, len(valid)), random_state=SEED)
            source_k = np.array([len(ids) for ids in tokenizer(sample["text"].tolist(), verbose=False).input_ids])
            target_k = np.array([len(ids) for ids in tokenizer(text_target=sample["summary"].tolist(), verbose=False).input_ids])
            text_records.append({"Dataset": dataset, "Split": split, "Contoh lolos": len(valid),
                                 "Contoh diukur": len(sample), "Median token masukan": float(np.median(source_k)),
                                 "Median token target": float(np.median(target_k)),
                                 "Masukan >256": float((source_k > 256).mean()),
                                 "Target >72": float((target_k > 72).mean())})
if not text_records:
    text_records = replay.get("notebook_eda", {}).get("text", [])
if text_records:
    token_stats = pd.DataFrame(text_records)
    display(token_stats.style.format({"Masukan >256": "{:.1%}", "Target >72": "{:.1%}"}))
    token_stats.set_index(["Dataset", "Split"])[["Median token masukan", "Median token target"]].plot.barh(
        color=PALETTE[:2], figsize=(8, 3.4), title="Panjang masukan dan ringkasan referensi")
    plt.show()
for split, frame in tables["Coding Video Mix"].items():
    print(split, frame["source"].value_counts().to_dict())

- IndoSum mengajarkan pola peringkasan artikel. Coding Video Mix memberi contoh instruksi, kode dan detail yang perlu dipertahankan.
- Setiap model dilatih pada IndoSum dan Coding Video Mix secara terpisah. Ini membantu melihat perbedaan antara berita dan tutorial.
- Contoh sintetis proyek hanya masuk latih. Pasangan CrossTask berbahasa Inggris tersaring oleh syarat `id_ID → id_ID`. Data sintetis regresi OCR tetap dipakai untuk uji terkontrol terpisah.
- Persentase pemotongan berasal dari token SentencePiece mBART dan dibaca sebagai perkiraan pada sampel.

## 05. Audio ke representasi log-Mel

In [ ]:
from scripts.notebook_asr import SAMPLE_RATE, read_audio

nss_train = tables["NSS-ID"].get("train")
audio_example = next((BASE / "nss-id" / item for item in nss_train["audio_path"]
                      if (BASE / "nss-id" / item).is_file()), None) if nss_train is not None else None
if audio_example and shutil.which("ffmpeg"):
    from transformers import WhisperFeatureExtractor

    wave = read_audio(audio_example)
    matrix = WhisperFeatureExtractor()(wave, sampling_rate=SAMPLE_RATE, return_tensors="np").input_features[0]
    length = min(matrix.shape[1], round(len(wave) / 160))
    fig, axes = plt.subplots(1, 2, figsize=(10, 3.1), layout="constrained")
    axes[0].plot(np.arange(len(wave)) / SAMPLE_RATE, wave, color=PALETTE[1], linewidth=0.7)
    axes[0].set(title="Audio mono pada 16 kHz", xlabel="Detik", ylabel="Amplitudo")
    axes[1].imshow(matrix[:, :length], origin="lower", aspect="auto", cmap="magma")
    axes[1].set(title="Spektrogram log-Mel", xlabel="Frame aktif", ylabel="Pita Mel")
    plt.show()
    print(f"{len(wave):,} sampel | {len(wave) / SAMPLE_RATE:.2f} detik | bentuk fitur {matrix.shape} | {length} frame aktif")

- FFmpeg mengambil jalur audio dan menyamakannya ke mono 16 kHz. Gelombang berisi 16.000 sampel per detik.
- Whisper base menghitung STFT dengan jendela 400 sampel dan langkah 160 sampel. Energi dipetakan ke 80 pita Mel, lalu diubah ke skala log. Bentuk masukannya 80 × 3.000 untuk jendela 30 detik.
- Encoder membaca fitur akustik. Decoder memakai konteks encoder untuk menebak token Indonesia dan menghasilkan penanda waktu saat transkripsi. [Paper Whisper](https://arxiv.org/abs/2212.04356)

## 06. Pelatihan model transkripsi Whisper

In [ ]:
LATIH_WHISPER = True
EPOCH_WHISPER = 6
AKUMULASI_WHISPER = 1
PATIENCE_WHISPER = 0

TRAIN_WHISPER = LATIH_WHISPER if IS_COLAB else LOCAL_MODE != "replay"
WHISPER_PROFILE = "pilot" if IS_COLAB else LOCAL_MODE
if EPOCH_WHISPER < 1 or AKUMULASI_WHISPER < 1 or PATIENCE_WHISPER < 0:
    raise ValueError("Epoch dan akumulasi minimal 1. Patience minimal 0.")

if TRAIN_WHISPER:
    for split in SPLITS:
        if not (BASE / "whisper_video" / f"{split}.csv").is_file():
            subprocess.run([sys.executable, "-m", "scripts.prepare_whisper_video", "--split", split], check=True)
whisper_dir = RUN / "whisper"
if TRAIN_WHISPER:
    execute("scripts.notebook_training", "whisper", "--profile", WHISPER_PROFILE, "--output", whisper_dir,
            "--device", WHISPER_DEVICE, "--epochs", EPOCH_WHISPER, "--accumulation", AKUMULASI_WHISPER,
            "--patience", PATIENCE_WHISPER)
    whisper_report = read_json(whisper_dir / "report.json")
else:
    whisper_report = replay.get("mixed_whisper")

if whisper_report:
    print("Hasil sesi ini" if TRAIN_WHISPER else "Contoh hasil uji cepat tersimpan")
    display(pd.DataFrame(whisper_report["splits"]).T)
    print(f"{whisper_report['base']} | {whisper_report['mode']} | {whisper_report['device']} | "
          f"{len(whisper_report['history'])} epoch dijalankan | epoch terpilih {whisper_report['best_epoch']} | LR {whisper_report['learning_rate']:g}")
    stages = pd.DataFrame(whisper_report["history"]).set_index("epoch")
    fig, axes = plt.subplots(1, 2, figsize=(9, 2.8), layout="constrained")
    stages["loss"].plot(ax=axes[0], marker="o", color=PALETTE[1], title="Cross-entropy latih")
    stages[["macro_wer", "macro_cer"]].plot(ax=axes[1], marker="o", color=PALETTE[:2], title="Validasi per epoch")
    axes[1].yaxis.set_major_formatter(PercentFormatter(1))
    plt.show()
else:
    print("Laporan pelatihan campuran belum tersedia.")

- Whisper menyediakan transkrip agar penjelasan lisan bisa masuk ke ringkasan. Korpus latih diacak ulang setiap epoch. Profil `pilot` mengambil hingga 16 klip latih, 8 validasi, dan 12 uji per korpus.
- Enam epoch adalah titik awal untuk mengamati WER dan CER validasi. Ubah `EPOCH_WHISPER` untuk mencoba lebih banyak. `PATIENCE_WHISPER = 0` menjalankan semua epoch. Isi `2` untuk berhenti setelah dua epoch tanpa perbaikan validasi.
- Checkpoint terpilih punya rata-rata WER validasi terendah, lalu CER sebagai pemecah seri. Epoch terpilih bisa lebih awal dari jumlah epoch yang dijalankan.

## 07. Video utuh dan hasil transkripsi

In [ ]:
VIDEO_MP4 = ""  # Isi dengan path MP4 untuk uji video baru.

from urllib.parse import parse_qs, urlsplit
from IPython.display import YouTubeVideo
from src.youtube import youtube_watch_url

raw_url = os.getenv("NLP_MULTIMODAL_YOUTUBE_URL", "").strip()
test_video = VIDEO_MP4.strip() or os.getenv("NLP_MULTIMODAL_VIDEO", "").strip()
if test_video and not Path(test_video).is_file():
    raise FileNotFoundError(f"MP4 belum tersedia di {test_video}.")
test_url = youtube_watch_url(raw_url) if raw_url else ("" if test_video else holdout[0]["video_url"])
test_id = parse_qs(urlsplit(test_url).query).get("v", [""])[0]
run_video = bool(raw_url or test_video or os.getenv("NLP_NOTEBOOK_FRESH_VIDEO") == "1"
                 or os.getenv("NLP_MULTIMODAL_RUN") == "1" or os.getenv("NLP_MULTIMODAL_TRANSCRIPT")
                 or os.getenv("NLP_NOTEBOOK_WHISPER_CHECKPOINT") or os.getenv("NLP_NOTEBOOK_MBART_WEIGHTS"))
prepared_path = RUN / "video_prepared.json"
archived_video = replay["whole_video"]

def video_time(value):
    minutes, seconds = divmod(int(value), 60)
    return f"{minutes:02d}:{seconds:02d}"

def video_stage(stage, *args):
    execute("scripts.notebook_video_ablation", stage, "--prepared", prepared_path, *args)
    return read_json(prepared_path)

if run_video:
    media = ROOT / "outputs/notebooks/multimodal/break.mp4"
    cache = ROOT / "outputs/notebooks/heldout-small" / f"{test_id}.json"
    flags = ["--url", test_url] if test_url else []
    if test_video or (test_id == holdout[0]["id"] and media.is_file()):
        flags += ["--video", test_video or media]
    transcript = os.getenv("NLP_MULTIMODAL_TRANSCRIPT", "").strip()
    if transcript:
        flags += ["--transcript", transcript]
    elif TRAIN_WHISPER or os.getenv("NLP_NOTEBOOK_WHISPER_CHECKPOINT"):
        flags += ["--whisper-checkpoint", os.getenv("NLP_NOTEBOOK_WHISPER_CHECKPOINT") or whisper_dir / "checkpoint"]
    elif test_id and cache.is_file():
        flags += ["--transcript", cache]
    transcribed = video_stage("transcribe", *flags)
    cues = transcribed["cues"]
else:
    print("Contoh hasil video pengembangan yang tersimpan")
    transcribed = archived_video["prepared"]
    cues = [{"start": cue["start"], "end": cue["end"], "transcript": cue["original"]}
            for part in transcribed["parts"] for cue in part["trace"]]
chosen = next((item for item in holdout if item["id"] == test_id), None)
if test_id:
    display(YouTubeVideo(test_id, width=560, height=315))
if chosen:
    display(Markdown(f"[{chosen['title']}]({test_url}) · [Kode pemateri]({chosen['code_url']})"))
print(f"{video_time(transcribed['duration'])} | {len(cues)} segmen | {transcribed['whisper']}")
if retry := transcribed.get("transcription_retry"):
    retry_model = retry["model"].replace(" adaptasi", " sesudah pelatihan tambahan")
    print(f"{retry_model} menghasilkan {retry['segments']} segmen. "
          f"{retry['reason']}. Audio dibaca ulang dengan {transcribed['whisper']}.")
display(pd.DataFrame(cues).head(5))

- FFprobe memeriksa durasi. Whisper atau cache dari video yang sama menyediakan teks dan waktu tiap segmen.
- Mode replay menampilkan jejak video nyata yang sudah tersimpan. Jika checkpoint adaptasi tidak memberi segmen atau satu segmen melampaui 45 detik, audio dibaca ulang dengan backend Faster Whisper dan pergantian dicatat.
- Skor video hanya tersedia bila ID sumber memiliki fakta dan detail acuan dalam manifest.

## 08. Cleaning, normalisasi dan pembagian konteks

In [ ]:
from collections import Counter
from src.transcript_cleaning import clean_utterance

prepared = video_stage("clean") if run_video else archived_video["prepared"]
trace = [cue for part in prepared["parts"] for cue in part["trace"]]
order = ("dirapikan", "dipertahankan", "pengisi mandiri", "duplikat tumpang tindih")
counts_clean = pd.Series(Counter(cue["status"] for cue in trace)).reindex(order, fill_value=0)
fillers = Counter(word.casefold() for cue in trace for word in cue.get("fillers_removed", []))
fig, axes = plt.subplots(1, 2, figsize=(10, 3), layout="constrained")
counts_clean.sort_values().plot.barh(ax=axes[0], color=PALETTE[1], title="Tindakan per segmen")
axes[0].set_xlabel("Segmen")
if fillers:
    pd.Series(fillers).sort_values().plot.barh(ax=axes[1], color=PALETTE[0], title="Pengisi yang dibuang")
    axes[1].set_xlabel("Kemunculan")
else:
    axes[1].text(0.5, 0.5, "Tidak ada pengisi yang dibuang", ha="center")
    axes[1].axis("off")
plt.show()

def show_text_table(rows):
    table = pd.DataFrame(rows)
    if not table.empty:
        display(table.style.format(escape="html").set_properties(
            **{"white-space": "pre-wrap", "overflow-wrap": "anywhere", "text-align": "left",
               "vertical-align": "top", "max-width": "34ch"}
        ))

changed = sorted(
(cue for cue in trace if cue["status"] != "dipertahankan"),
                 key=lambda cue: len(cue["original"]))[:3]
show_text_table([{"Mulai": video_time(cue["start"]), "Sebelum": cue["original"],
                       "Sesudah": cue["cleaned"], "Pengisi dibuang": ", ".join(cue.get("fillers_removed", []))}
                      for cue in changed])
examples = (
    "Nah kita buka file CSV.",
    "Lalu, apa ya, kita jalankan lagi.",
    "Apa yang terjadi kalau input kosong?",
    "Statusnya oke.",
    "Pakai 4, eh, maksudnya 5.",
    'Jalankan print("oke") untuk cek hasil.',
)
display(Markdown("**Contoh terkontrol untuk memeriksa konteks**"))
show_text_table([{"Sebelum": text, "Sesudah": clean_utterance(text)[0]} for text in examples])

display(Markdown(f"Dari **{len(trace)} segmen**, aturan lokal menghapus **{sum(fillers.values())} kemunculan pengisi**. "
                 "Pada tabel, lihat kata yang berubah sebelum membandingkan versi Gemini di bagian 10."))

Grafik menghitung perubahan yang benar-benar dilakukan pada transkrip. Tabel membungkus teks supaya bagian akhir kalimat tetap terbaca. Contoh terkontrol menunjukkan mengapa kata yang sama bisa dihapus saat menjadi jeda dan dipertahankan saat membawa makna. Perbedaan ini penting sebelum Gemini membaca bagian yang masih rancu pada tahap berikutnya.

## 09. Bukti visual dari OCR lokal

In [ ]:
from base64 import b64decode
from io import BytesIO
from PIL import Image

if run_video:
    roi = os.getenv("NLP_MULTIMODAL_ROI",
                    "0.225 0.115 0.61 0.90" if test_id == holdout[0]["id"] else "0 0 1 1").split()
    prepared = video_stage("ocr", "--roi", *roi)
previews = [part["preview"] for part in prepared["parts"] if "preview" in part]
if previews:
    fig, axes = plt.subplots(1, len(previews), figsize=(11, 2.5), squeeze=False, layout="constrained")
    for axis, item in zip(axes[0], previews, strict=True):
        with Image.open(BytesIO(b64decode(item["image"]))) as image:
            axis.imshow(image)
        axis.set_title(video_time(item["second"]))
        axis.axis("off")
    plt.show()
frame_table = pd.DataFrame([{"Bagian": part["id"], "Frame": len(part["frames"]),
                             "Baris OCR": sum(len(frame["lines"]) for frame in part["frames"])}
                            for part in prepared["parts"]])
display(frame_table)
findings = pd.DataFrame(prepared["actions"])
if findings.empty:
    print("OCR lokal belum menemukan pola aksi yang lolos pemeriksaan antar-frame.")
else:
    display(findings[["part", "loop", "condition", "increment", "uses_input"]])

- FFmpeg mengambil beberapa frame setiap bagian. Area editor dipotong, diubah ke grayscale dan diperbesar sebelum Tesseract membaca teks.
- Sistem menyimpan hasil baca serta waktu frame. Pola kondisi dan instruksi kode yang berulang pada lebih dari satu frame menjadi bukti yang dapat dipakai untuk memperjelas ucapan.
- Tabel menampilkan cakupan frame dan baris. Cuplikan layar memperlihatkan dampak ukuran teks, crop dan kualitas video terhadap OCR.

## 10. Gemini multimodal sebelum ringkasan

In [ ]:
PAKAI_GEMINI = False
BATAS_GEMINI = 4

GEMINI_LIMIT = BATAS_GEMINI if PAKAI_GEMINI else int(os.getenv("NLP_GEMINI_MAX_REQUESTS", "0"))
if GEMINI_LIMIT < 0 or (PAKAI_GEMINI and GEMINI_LIMIT == 0):
    raise ValueError("Atur batas permintaan Gemini ke angka positif saat Gemini diaktifkan.")
if PAKAI_GEMINI and not run_video:
    raise ValueError("Isi VIDEO_MP4 di bagian 07 dan ulangi bagian 07 sampai 10 untuk memakai Gemini.")
if IS_COLAB and run_video and GEMINI_LIMIT:
    from google.colab import userdata
    from google.colab.userdata import NotebookAccessError, SecretNotFoundError

    try:
        gemini_key = userdata.get("GEMINI_API_KEY")
    except (NotebookAccessError, SecretNotFoundError) as exc:
        raise RuntimeError("Tambahkan GEMINI_API_KEY di Secrets Colab dan aktifkan akses untuk notebook ini.") from exc
    if not gemini_key or not gemini_key.strip():
        raise ValueError("Isi GEMINI_API_KEY di Secrets Colab.")
    os.environ["GEMINI_API_KEY"] = gemini_key
    del gemini_key

if run_video:
    prepared = video_stage("gemini", "--gemini-max-requests", GEMINI_LIMIT)
vision = prepared["gemini"]
from html import escape
from IPython.display import HTML

display(Markdown(f"**Status:** {vision['status']}  \n"
                 f"**Model:** {vision.get('model') or '—'}  \n"
                 f"**Permintaan:** {vision['requests']} · **Bukti kode:** {len(vision['verified'])}"))
for item in vision["verified"]:
    code = escape(item["code"])
    quote = escape(item["quote"])
    frames = escape(", ".join(item["frame_ids"]))
    display(HTML(
        f"<div style='margin:0 0 12px;max-width:70ch'><strong>Bagian {item['part']}</strong>"
        f" · Frame {frames}<pre style='white-space:pre-wrap;margin:6px 0'><code>{code}</code></pre>"
        f"<details><summary>Baca kutipan transkrip</summary><p style='margin:8px 0'>{quote}</p></details></div>"
    ))

context_report = prepared.get("context_cleaning") if run_video else replay.get("contextual_cleaning")
if context_report:
    proposals = context_report.get("proposals", 0)
    accepted = len(context_report["changes"])
    print(context_report["status"], "| kandidat:", context_report.get("candidate_count", 0),
          "| usulan:", proposals, "| diterima:", accepted)
    show_text_table([{"Mulai": video_time(item["start"]), "Lokal": item["local"], "Gemini": item["context"],
                      "Perubahan": item["kind"]} for item in context_report["changes"]])
    if not run_video:
        display(Markdown("Audit Gemini ini memakai cuplikan video tersimpan dari pengujian lokal."))
fixture = replay.get("contextual_fixture")
if fixture:
    display(Markdown("**Uji terkontrol cleaning kontekstual**"))
    print(fixture["status"], "| diterima:", len(fixture["changes"]))
    show_text_table([{"Lokal": item["local"], "Gemini": item["context"], "Perubahan": item["kind"]}
                     for item in fixture["changes"]])

Blok kode menampilkan bukti yang cocok dengan OCR. Buka kutipannya untuk memeriksa ucapan sumber. Pemeriksaan konteks memakai ucapan sebelum dan sesudah serta bukti kode tersebut ketika tersedia. Pemisahan ini berguna karena keberhasilan membaca kode belum tentu berarti ucapan perlu diubah. Jumlah usulan yang diterima memberi tahu apakah versi Gemini berbeda dari cleaning lokal. Contoh terkontrol menguji perubahan dengan bukti yang sengaja dibuat jelas, sementara angka pada video menunjukkan hasil nyata.

## 11. Token, ID dan embedding mBART

In [ ]:
example = "Gunakan break ketika penghitung mencapai 3."
token_example = replay.get("embedding_example")
if tokenizer and not token_example:
    import torch
    from transformers import AutoConfig

    model_config = AutoConfig.from_pretrained(mbart_path, local_files_only=True)
    ids = tokenizer(example).input_ids
    token_table = pd.DataFrame({"Token": tokenizer.convert_ids_to_tokens(ids), "ID kosakata": ids})
    with torch.no_grad():
        weights = torch.load(mbart_path / "pytorch_model.bin", mmap=True, weights_only=True, map_location="cpu")
        embedding = weights["model.shared.weight"][ids, :4].clone().numpy()
        del weights
    for index in range(embedding.shape[1]):
        token_table[f"E[{index}]"] = embedding[:, index]
    token_example = {"rows": token_table.to_dict("records"), "d_model": model_config.d_model,
                     "heads": model_config.encoder_attention_heads, "language_id": tokenizer.lang_code_to_id["id_ID"]}
if token_example:
    display(pd.DataFrame(token_example["rows"]).round(4))
    print(f"ID bahasa id_ID = {token_example['language_id']} | dimensi embedding per token = "
          f"{token_example['d_model']} | attention heads = {token_example['heads']}")
else:
    print("Cache tokenizer dan checkpoint diperlukan untuk menampilkan embedding aktual.")

- SentencePiece memecah teks menjadi subword, lalu tokenizer memetakan tiap token ke ID. Tag `id_ID` dipakai pada masukan dan keluaran.
- Setiap ID mengambil satu baris dari matriks embedding pralatih. Tabel menunjukkan empat komponen aktual dari vektor penuh. Encoder menambahkan informasi posisi agar kata yang sama di posisi berbeda punya konteks berbeda.
- Batas latihan adalah 512 token sumber dan 96 token target untuk IndoSum, serta 256 dan 72 untuk Coding Video Mix. Bagian panjang dibatasi saat encoding. [Checkpoint mBART-large-50](https://huggingface.co/facebook/mbart-large-50)

## 12. Perhitungan self-attention, causal mask dan cross-attention

In [ ]:
def attention(query, key, value, *, causal=False):
    scores = query @ key.T / np.sqrt(query.shape[1])
    if causal:
        scores = np.where(np.triu(np.ones(scores.shape, dtype=bool), 1), -np.inf, scores)
    stabilized = np.exp(scores - scores.max(axis=1, keepdims=True))
    weights = stabilized / stabilized.sum(axis=1, keepdims=True)
    return weights, weights @ value

source = np.array([[1.0, 0.0], [0.5, 1.0], [0.0, 1.5]])
target_demo = np.array([[1.0, 0.5], [0.0, 1.0]])
names_src = ["kondisi", "angka 3", "break"]
names_tgt = ["Hentikan", "perulangan"]
views = [
    ("Self-attention encoder", source, source, source, False, names_src, names_src),
    ("Self-attention decoder", target_demo, target_demo, target_demo, True, names_tgt, names_tgt),
    ("Cross-attention", target_demo, source, source, False, names_tgt, names_src),
]
fig, axes = plt.subplots(1, 3, figsize=(11, 3.1), layout="constrained")
for axis, (title, q, k, v, mask, rows, columns) in zip(axes, views, strict=True):
    weights, context = attention(q, k, v, causal=mask)
    axis.imshow(weights, vmin=0, vmax=1, cmap="Blues")
    axis.set(xticks=range(len(columns)), xticklabels=columns, yticks=range(len(rows)), yticklabels=rows, title=title)
    for y, x in np.ndindex(weights.shape):
        axis.text(x, y, f"{weights[y, x]:.2f}", ha="center", va="center",
                  color="white" if weights[y, x] > 0.55 else "#202631")
plt.show()
print("Contoh vektor konteks cross-attention untuk token pertama", context[0].round(3))

- **Encoder self-attention** membentuk Q, K dan V dari token sumber, sehingga kata seperti “kondisi” dapat merujuk ke “angka 3” dan “break”.
- **Decoder self-attention** memakai causal mask. Skor token yang berada di depan diganti dengan $-infty$, sehingga bobotnya menjadi nol.
- **Cross-attention** mengambil Q dari decoder, serta K dan V dari encoder. Bobotnya $operatorname{softmax}(QK^	op/sqrt{d_k})$ dan hasil konteksnya adalah bobot dikali V. Setiap baris dijumlahkan menjadi 1.
- Heatmap memakai vektor contoh 2 dimensi agar hitungannya terbaca. Bobot mBART diperoleh dari proyeksi terlatih di setiap head. [Paper Transformer](https://arxiv.org/abs/1706.03762)

## 13. Pelatihan tiga model peringkas

In [ ]:
LATIH_PERINGKAS = True
EPOCH_PERINGKAS = 6
AKUMULASI_PERINGKAS = 4
PATIENCE_PERINGKAS = 0
INDOSUM_LATIH = 500
INDOSUM_VALIDASI = 50
INDOSUM_UJI = 100

TRAIN_SUMMARY = LATIH_PERINGKAS if IS_COLAB else LOCAL_MODE != "replay"
SUMMARY_PROFILE = "pilot" if IS_COLAB else LOCAL_MODE
if min(EPOCH_PERINGKAS, AKUMULASI_PERINGKAS, INDOSUM_LATIH, INDOSUM_VALIDASI, INDOSUM_UJI) < 1 or PATIENCE_PERINGKAS < 0:
    raise ValueError("Epoch, akumulasi, dan ukuran sampel minimal 1. Patience minimal 0.")

from scripts.compare_partial_seq2seq import MODELS_IN_COMPARISON, comparison_reports

model_labels = {"mbart": "mBART Large", "mt5": "mT5 Small", "indobart-v2": "IndoBART v2"}
dataset_labels = {"indosum": "IndoSum", "coding_video_mix": "Coding Video Mix"}
comparison_dir = RUN / "three_models"
if TRAIN_SUMMARY:
    data_options = ["--indosum-limits", INDOSUM_LATIH, INDOSUM_VALIDASI, INDOSUM_UJI] if SUMMARY_PROFILE == "pilot" else []
    for model_key in MODELS_IN_COMPARISON:
        execute("scripts.compare_partial_seq2seq", "--model", model_key, "--output", comparison_dir,
                "--profile", SUMMARY_PROFILE, "--device", MBART_DEVICE, "--download",
                "--epochs", EPOCH_PERINGKAS, "--accumulation", AKUMULASI_PERINGKAS,
                "--patience", PATIENCE_PERINGKAS, *data_options, python=MBART_PYTHON)
    model_reports = comparison_reports(comparison_dir)
else:
    override = os.getenv("NLP_SEQ2SEQ_REPORT_DIR", "").strip()
    model_reports = replay["partial_seq2seq"]
    if override:
        model_reports = comparison_reports(Path(override))
    else:
        complete = []
        for directory in (ROOT / "outputs/notebooks").glob("seq2seq-partial-*"):
            try:
                reports = comparison_reports(directory)
            except (OSError, KeyError, ValueError):
                continue
            files = [directory / row["dataset"] / row["model"] / "report.json" for row in reports]
            complete.append((min(file.stat().st_mtime for file in files), reports))
        if complete:
            model_reports = max(complete, key=lambda item: item[0])[1]

print("Hasil sesi ini" if TRAIN_SUMMARY else
      "Laporan tersimpan" if model_reports is not replay["partial_seq2seq"] else "Contoh hasil uji cepat tersimpan")
history_rows = [
    {"Dataset": dataset_labels[row["dataset"]], "Model": model_labels[row["model"]],
     "Epoch": train["epoch"], "Loss latih": round(train["loss"], 4),
     "ROUGE-L validasi": train["validation"]["rougeL"] * 100}
    for row in model_reports for train in row["history"]
]
history = pd.DataFrame(history_rows)
fig, axes = plt.subplots(1, 2, figsize=(10, 2.9), layout="constrained")
for axis, (dataset, label) in zip(axes, dataset_labels.items(), strict=True):
    for model_key, color in zip(MODELS_IN_COMPARISON, PALETTE, strict=True):
        name = model_labels[model_key]
        values = history[(history["Dataset"] == label) & (history["Model"] == name)]
        axis.plot(values["Epoch"], values["ROUGE-L validasi"], marker="o", color=color, label=name)
    axis.set(title=label, xlabel="Epoch", ylabel="ROUGE-L validasi (0–100)")
    axis.set_xticks(sorted(history.loc[history["Dataset"] == label, "Epoch"].unique()))
axes[1].legend(frameon=False, fontsize=8)
plt.show()
display(history.round(2))
configuration = model_reports[0]
print(f"{configuration['profile']} | maks. {configuration['epochs_limit']} epoch | "
      f"batch fisik {configuration['batch_size']} | akumulasi gradien {configuration['accumulation']}")
if TRAIN_SUMMARY:
    candidate = comparison_dir / "coding_video_mix/mbart/deployment.json"
    if candidate.is_file():
        print(f"Kandidat Streamlit lolos validasi Coding Video Mix: {candidate}")
    else:
        print("Validasi Coding Video Mix mempertahankan mBART bawaan untuk Streamlit.")

- Satu epoch berarti satu putaran pada seluruh contoh latih. Enam epoch memberi titik awal untuk melihat kurva validasi. Ubah `EPOCH_PERINGKAS` jika kurva masih membaik. `PATIENCE_PERINGKAS = 0` menjalankan semua epoch. Isi `2` untuk berhenti setelah dua epoch tanpa perbaikan.
- Run Colab memakai 500 artikel latih, 50 validasi, dan 100 uji IndoSum. Coding Video Mix memakai 46 latih, 6 validasi, dan 9 uji berbahasa Indonesia. Log memperlihatkan unduhan awal, latihan, dan evaluasi.
- mBART dan IndoBART v2 memperbarui proyeksi Q dan V pada cross-attention decoder terakhir. mT5 memakai LoRA pada proyeksi Q/V dan kepala keluaran. Ketiga model memakai split dan metrik yang sama dengan strategi adaptasi masing-masing.
- Checkpoint dipilih dengan ROUGE-L validasi. Recall detail kode menjadi pemecah seri pada Coding Video Mix. Skor dari data uji dihitung setelah checkpoint terpilih.

## 13B. Optimasi mBART Large pada IndoSum

Bagian ini menguji apakah mBART bisa menghasilkan ringkasan berita yang lebih baik. Konfigurasi generasi diuji pada split validasi dengan batas 96, 160, atau 192 token dan 2 atau 5 beam. Kandidatnya mencakup mBART pralatih, checkpoint mBART bagian 13 jika tersedia, dan LoRA di empat lapisan decoder terakhir. Setelan awal LoRA memakai hingga tiga epoch, *learning rate* `1e-4`, dan 2.000 artikel latih. Setelan beam yang lebih lebar mengikuti arah eksperimen IndoNLG. [Protokol evaluasi IndoNLG](https://aclanthology.org/2021.emnlp-main.699.pdf#page=6)

Pemenang dipilih dengan **ROUGE-L validasi**, lalu diuji pada artikel yang sama dengan perbandingan utama saat jumlah validasi dan uji sama. Perbandingan utama di bagian 16 tetap menyediakan hasil dengan anggaran eksperimen awal. Bagian 13B memberi mBART putaran optimasi tambahan pada IndoSum. Uji Coding Video Mix menjadi dasar terpisah untuk pemilihan model tutorial.

Untuk optimasi cepat tanpa latihan LoRA, ubah `LATIH_LORA_MBART = False`. Jika kamu punya hasil mBART dari run Colab sebelumnya, simpan `report.json` dan `selected.pt` di folder yang sama, lalu isi `LAPORAN_MBART_BAGIAN_13` dengan path ke JSON. Notebook memeriksa revisi model dan sidik data sebelum memakai hasil itu.

In [ ]:
OPTIMASI_MBART_INDOSUM = True
LATIH_LORA_MBART = True
INDOSUM_LATIH_MBART = 2000
EPOCH_LORA_MBART = 3
AKUMULASI_LORA_MBART = 8
PATIENCE_LORA_MBART = 1
LAPORAN_MBART_BAGIAN_13 = ""

mbart_opt_dir = RUN / "mbart-indosum-optimized"
mbart_opt_report = mbart_opt_dir / "report.json"
validation_count = globals().get("INDOSUM_VALIDASI", 50)
test_count = globals().get("INDOSUM_UJI", 100)

if IS_COLAB and OPTIMASI_MBART_INDOSUM and not mbart_opt_report.is_file():
    prior_report = Path(LAPORAN_MBART_BAGIAN_13).expanduser() if LAPORAN_MBART_BAGIAN_13.strip() else None
    if prior_report is None and globals().get("TRAIN_SUMMARY") and SUMMARY_PROFILE == "pilot":
        prior_report = comparison_dir / "indosum/mbart/report.json"
    extra = ["--prior-report", prior_report] if prior_report is not None else []
    if not LATIH_LORA_MBART:
        extra.append("--skip-lora")
    execute("scripts.optimize_mbart_indosum", "--output", mbart_opt_dir, "--device", MBART_DEVICE,
            "--counts", INDOSUM_LATIH_MBART, validation_count, test_count, "--epochs", EPOCH_LORA_MBART,
            "--accumulation", AKUMULASI_LORA_MBART, "--patience", PATIENCE_LORA_MBART, *extra, python=MBART_PYTHON)

if mbart_opt_report.is_file():
    optimized = read_json(mbart_opt_report)
    def rouge_points(metrics):
        return {label: round(metrics[key] * 100, 2)
                for key, label in (("rouge1", "ROUGE-1"), ("rouge2", "ROUGE-2"), ("rougeL", "ROUGE-L"))}

    display(Markdown("### Pemilihan konfigurasi pada validasi"))
    validation_table = pd.DataFrame([
        {"Checkpoint": row["method"], "Maks. token": row["tokens"], "Beam": row["beams"],
         **rouge_points(row["validation"])} for row in optimized["candidates"]
    ]).sort_values(["ROUGE-L", "ROUGE-2", "Beam", "Maks. token"], ascending=[False, False, True, True])
    display(validation_table.style.hide(axis="index").format(precision=2))

    winner = optimized["selected"]
    display(Markdown(f"**Terpilih** · {winner['method']} · {winner['tokens']} token · {winner['beams']} beam"))
    test_table = [
        {"Model dan konfigurasi": "mBART pralatih · 96 token, 2 beam", **rouge_points(optimized["baseline_test"])},
        {"Model dan konfigurasi": "mBART terpilih pada validasi", **rouge_points(optimized["selected_test"])},
    ]
    selected_ids = [item["id"] for item in optimized["selected_test_predictions"]]
    comparison_names = {"mbart": "mBART Large", "mt5": "mT5 Small", "indobart-v2": "IndoBART v2"}
    for row in globals().get("model_reports", ()):
        prior_ids = [item["id"] for item in row.get("test_predictions", ())]
        if row["dataset"] == "indosum" and prior_ids == selected_ids:
            test_table.append({"Model dan konfigurasi": f"{comparison_names[row['model']]} · bagian 13",
                               **rouge_points(row["test"])})
    display(Markdown(f"### Hasil pada {len(selected_ids)} artikel uji IndoSum"))
    display(pd.DataFrame(test_table).style.hide(axis="index").format(precision=2))
    delta = (optimized["selected_test"]["rougeL"] - optimized["baseline_test"]["rougeL"]) * 100
    print(f"Perubahan ROUGE-L uji terhadap mBART pralatih: {delta:+.2f} poin")
    print(f"Laporan: {mbart_opt_report}")
else:
    print("Hasil optimasi mBART akan tampil setelah eksperimen dijalankan di Colab.")

## 14. Inferensi A/B/C pada video yang sama

In [ ]:
ACUAN_VIDEO_JSON = ""  # Isi jika MP4 punya acuan manual.

variant_labels = {"transcript": "A · Transkrip", "ocr": "B · Transkrip + OCR",
                  "gemini": "C · Transkrip + OCR + Gemini"}
if run_video:
    outcome_path = RUN / "video_summary.json"
    flags = ["--output", outcome_path, "--device", MBART_DEVICE, "--download"]
    if ACUAN_VIDEO_JSON.strip():
        flags += ["--reference", ACUAN_VIDEO_JSON.strip()]
    if TRAIN_SUMMARY or os.getenv("NLP_NOTEBOOK_MBART_WEIGHTS"):
        flags += ["--mbart-weights", os.getenv("NLP_NOTEBOOK_MBART_WEIGHTS") or comparison_dir / "coding_video_mix/mbart/selected.pt"]
    execute("scripts.notebook_video_ablation", "summarize", "--prepared", prepared_path, *flags, python=MBART_PYTHON)
    outcome = read_json(outcome_path)
else:
    outcome = archived_video["outcome"]

variants = outcome["variants"]
display(pd.DataFrame([{"Jalur": name, "Jumlah poin": len(variants[key]["rows"]) if key in variants else None,
                       "Detik": round(variants[key]["seconds"], 2) if key in variants else None,
                       "Status": "Selesai" if key in variants else vision["status"]}
                      for key, name in variant_labels.items()]))
checkpoint_label = outcome["checkpoint"]
if run_video:
    checkpoint_label = AFTER if "--mbart-weights" in flags else BEFORE
elif checkpoint_label.startswith("facebook/") and " ? " not in checkpoint_label:
    checkpoint_label = BEFORE
print("mBART", "|", checkpoint_label, "|", outcome["precision"])

- A memakai transkrip bersih. B menambahkan bukti OCR lokal. C memakai bukti tambahan Gemini yang terverifikasi. Semua jalur yang tersedia memakai model dan aturan decoding yang sama.
- mBART menghasilkan kandidat per langkah dengan beam search 2. Pemeriksaan sumber menjaga angka dan ekspresi kode. Saat kandidat gagal, sistem dapat memakai rumusan yang didukung sumber. Tidak setiap poin akhir merupakan keluaran mentah model.
- Hasil replay video mempunyai identitas checkpoint sendiri. Untuk memakai checkpoint terbaru, aktifkan pelatihan dan inferensi video dalam satu sesi.

## 15. Evaluasi transkripsi dan kesalahan pengenalan

In [ ]:
if whisper_report:
    stage_labels = {"base": BEFORE, "adapted": AFTER}
    errors = pd.DataFrame([{"Dataset": name, "Tahap model": stage_labels[key], **values}
                           for name, test in whisper_report["test"].items() for key, values in test.items()])
    display(errors.style.format({"wer": "{:.1%}", "cer": "{:.1%}"}))
    fig, axes = plt.subplots(1, 2, figsize=(9, 3.2), layout="constrained")
    for axis, metric in zip(axes, ("wer", "cer"), strict=True):
        errors.pivot(index="Dataset", columns="Tahap model", values=metric)[[BEFORE, AFTER]].plot.bar(ax=axis, rot=0, color=PALETTE[:2], title=metric.upper())
        axis.yaxis.set_major_formatter(PercentFormatter(1))
    plt.show()
    comparison = errors.groupby("Tahap model")[["wer", "cer"]].mean().reindex([BEFORE, AFTER])
    display(comparison.style.format("{:.1%}"))

from scripts.notebook_asr import risk_roc

audit = pd.DataFrame(replay["asr"]["NSS-ID"]["audit"]).rename(columns={"wer_adaptasi": "wer"})
roc = risk_roc(audit, threshold=0.25)
if roc:
    fpr, tpr, auc, n = roc
    fig, axis = plt.subplots(figsize=(4.5, 3.25), layout="constrained")
    axis.plot(fpr, tpr, color=PALETTE[1], label=f"AUC {auc:.3f}")
    axis.plot([0, 1], [0, 1], color=PALETTE[0], linestyle=":")
    axis.set(title=f"ROC risiko transkrip · arsip NSS-ID, n={n}", xlabel="False positive rate",
             ylabel="True positive rate")
    axis.legend(frameon=False)
    plt.show()

- WER dan CER menghitung (substitusi + penghapusan + penyisipan) dibagi jumlah unit acuan. WER memakai kata dan CER memakai karakter. Skor lebih rendah lebih baik.
- Subtitle otomatis pada Audio coding memberi acuan perkiraan. Skor identik pada subset kecil belum cukup untuk menyimpulkan manfaat adaptasi.
- ROC adalah analisis tambahan dari arsip NSS-ID, terpisah dari eksperimen gabungan. Label risiko memakai WER ≥ 25% dan skor risiko memakai negatif log-probabilitas rerata. Kurva mengukur pemisahan klip berisiko oleh skor tersebut.
- **Sebelum pelatihan tambahan** memakai model pralatih. **Sesudah pelatihan tambahan** memakai model yang telah dilatih lagi pada split latih proyek. Keduanya dinilai pada data uji yang sama.

## 16. Evaluasi model peringkas dan pembanding

In [ ]:
scores = ("rouge1", "rouge2", "rougeL")
columns = dict(zip(scores, ("ROUGE-1", "ROUGE-2", "ROUGE-L"), strict=True))

def as_points(score):
    return round(score * 100, 2)

display(Markdown("**Perbandingan tiga model peringkas · skala ROUGE 0–100**"))
for dataset, label in dataset_labels.items():
    group = [row for row in model_reports if row["dataset"] == dataset]
    display(Markdown(f"**{label}** · {group[0]['source_tokens']} token sumber dan "
                     f"{group[0]['target_tokens']} token keluaran"))
    display(pd.DataFrame([{
        "Model": model_labels[row["model"]], "Latih": row["counts"]["train"],
        "Validasi": row["counts"]["validation"], "Uji": row["counts"]["test"],
        "Batch fisik": row["batch_size"], "Epoch dijalankan": len(row["history"]),
        "Epoch terpilih": row["best_epoch"],
        **{metric_label: as_points(row["test"][key]) for key, metric_label in columns.items()},
    } for row in group]).style.hide(axis="index").format(precision=2))

indo = [row for row in model_reports if row["dataset"] == "indosum"]
display(Markdown("**Pengaruh latihan IndoSum pada ketiga model**"))
display(pd.DataFrame([{
    "Model": model_labels[row["model"]], "Epoch terpilih": row["best_epoch"],
    f"ROUGE-L ? {BEFORE}": as_points(row["base"]["rougeL"]),
    f"ROUGE-L ? {AFTER}": as_points(row["test"]["rougeL"]),
    "Selisih ROUGE-L": as_points(row["test"]["rougeL"] - row["base"]["rougeL"]),
    f"Loss ? {BEFORE}": round(row["base"]["loss"], 4),
    f"Loss ? {AFTER}": round(row["test"]["loss"], 4),
} for row in indo]).style.hide(axis="index").format(precision=2))
display(Markdown(f"Pada perbandingan IndoSum ini, setiap model "
                 f"belajar dari {indo[0]['counts']['train']} artikel, memilih epoch dengan "
                 f"{indo[0]['counts']['validation']} artikel validasi, lalu kedua checkpoint membaca "
                 f"{indo[0]['counts']['test']} artikel uji yang sama dengan ringkasan acuan yang sama. "
                 "Artikel uji dipakai untuk menghitung skor setelah latihan selesai."))

display(Markdown("### Hasil publikasi IndoNLG pada IndoSum"))
display(Markdown("Tabel ini memakai protokol fine-tuning dan evaluasi publikasi. "
                 "Hasil eksekusi notebook ada pada tabel di atas."))
display(pd.DataFrame([
    ["mBART Large", 74.65, 70.43, 72.54],
    ["mT5 Small", 74.04, 69.64, 71.89],
    ["IndoBART (generasi pertama)", 70.67, 65.59, 68.18],
], columns=["Model", *columns.values()]).style.hide(axis="index").format(precision=2))

mT5 pralatih Google memakai tugas pengisian bagian teks dan memerlukan fine-tuning terawasi untuk peringkasan. Pada uji cepat lama, keluarannya masih sering berisi token `<extra_id_0>`. Run baru menguji apakah keluaran membaik setelah LoRA pada data berlabel. Angka 71,89 berasal dari eksperimen IndoNLG dengan protokol tersendiri. [Kartu model mT5](https://huggingface.co/google/mt5-small) · [Tabel 5 IndoNLG](https://aclanthology.org/2021.emnlp-main.699.pdf)

## 17. Dampak OCR dan Gemini pada video utuh

In [ ]:
rows = []
for key, name in variant_labels.items():
    metrics = variants.get(key, {}).get("scores")
    if metrics:
        rows.append({"Jalur": name, "Cakupan fakta": metrics["coverage"], "Detail kode": metrics["detail_recall"],
                     "ROUGE-1": metrics["rouge1"], "ROUGE-2": metrics["rouge2"], "ROUGE-L": metrics["rougeL"]})
if rows:
    comparison_video = pd.DataFrame(rows).set_index("Jalur")
    display(comparison_video.reset_index().style.hide(axis="index").format(
        {metric: "{:.1%}" for metric in comparison_video.columns}, na_rep="—"))
    fig, axes = plt.subplots(1, 2, figsize=(10, 3.3), layout="constrained")
    for axis, columns in zip(axes, (["Cakupan fakta", "Detail kode"], ["ROUGE-1", "ROUGE-2", "ROUGE-L"]), strict=True):
        comparison_video[columns].dropna(how="all", axis=1).T.plot.bar(ax=axis, color=PALETTE, rot=0, ylim=(0, 1.05))
        axis.yaxis.set_major_formatter(PercentFormatter(1))
        axis.legend(fontsize=7, frameon=False)
    plt.show()
    if len(rows) > 1:
        display(Markdown("### Selisih terhadap jalur A dalam poin persentase"))
        delta = comparison_video.subtract(comparison_video.iloc[0]).iloc[1:].mul(100).reset_index()
        display(delta.style.hide(axis="index").format(
            {metric: "{:+.1f}" for metric in comparison_video.columns}, na_rep="—"))
else:
    print("Fakta acuan untuk video ini belum tersedia.")
if "gemini" not in variants:
    print("C tanpa skor,", vision["status"])

timing = {"Frame dan OCR": prepared["ocr_seconds"]}
for key, label in (("transcript_seconds", "Transkripsi atau baca cache"), ("cleaning_seconds", "Cleaning")):
    if key in prepared:
        timing[label] = prepared[key]
if vision["requests"]:
    timing["Panggilan Gemini"] = vision["seconds"]
timing.update({variant_labels[key]: item["seconds"] for key, item in variants.items()})
pd.Series(timing).sort_values().plot.barh(color=PALETTE[1], figsize=(8, 2.7), xlabel="Detik",
                                          title="Durasi tahap yang tercatat")
plt.show()

cases = replay["preprocessing_evaluation"]["cases"]
if os.getenv("NLP_PREPROCESSING_EVAL") == "1":
    audit_path = RUN / "preprocessing" / "report.json"
    execute("scripts.notebook_preprocessing_eval", "--output", audit_path)
    cases = read_json(audit_path)["cases"]
display(Markdown("**Cakupan uji regresi preprocessing**"))
def hits(entries, flag):
    return f"{sum(row[flag] for row in entries)}/{len(entries)}" if entries else "—"

display(pd.DataFrame([{"Sampel": item["title"], "Jenis": item["kind"], "Peran": item["role"],
                       "Fakta lama": hits(item["baseline"]["facts"], "covered"),
                       "Fakta revisi": hits(item["revised"]["facts"], "covered"),
                       "Kode lama": hits(item["baseline"]["details"], "present"),
                       "Kode revisi": hits(item["revised"]["details"], "present"),
                       "Status": item["status"]} for item in cases]))

- Cakupan fakta mengukur berapa fakta referensi yang dijumpai. Detail kode memeriksa bentuk yang lebih spesifik. Tabel selisih memakai A sebagai titik acuan.
- Perubahan pada satu video memberi bukti untuk video tersebut. Jalur C memerlukan respons yang terverifikasi agar dampak Gemini dapat dihitung.
- Uji regresi membandingkan aturan lama dan revisi pada video publik serta kasus sintetis, termasuk narasi bising, pembicaraan di luar domain, dan informasi yang hanya muncul di layar.
- Untuk menghitung skor pada MP4 baru, isi **ACUAN_VIDEO_JSON** dengan path JSON berisi `title`, `facts` dan `code_details`. Formatnya mengikuti satu entri pada [manifest acuan video](../samples/notebook_video_holdout.json). Buat acuan dari isi video sebelum melihat keluaran model.

## 18. Luaran akhir dan ekspor

In [ ]:
from html import escape
from IPython.display import HTML
from src.code_text import highlight_code

def timed_transcript(title, key):
    lines = "\n".join(f"[{video_time(cue['start'])}–{video_time(cue['end'])}] {cue[key].strip()}"
                      for cue in trace if cue[key].strip())
    return f"<details><summary>{escape(title)}</summary><pre style='white-space:pre-wrap'>{escape(lines)}</pre></details>"

display(HTML(timed_transcript("Transkrip asli lengkap", "original") +
             timed_transcript("Transkrip bersih lengkap", "cleaned")))
exports = []
for key, name in variant_labels.items():
    display(Markdown(f"### {name}"))
    if key not in variants:
        display(Markdown(f"Jalur belum menghasilkan bukti tambahan. {vision['status']}"))
        continue
    points = variants[key]["rows"]
    if not points:
        display(Markdown("Belum ada langkah yang cukup jelas dalam bukti yang tersedia."))
    else:
        items = "".join(f"<li>{highlight_code(point['summary'])}</li>" for point in points)
        display(HTML(f"<ul>{items}</ul>"))
    exports.extend({"variant": key, "point": index, "summary": point["summary"], "checkpoint": outcome["checkpoint"]}
                   for index, point in enumerate(points, 1))
if run_video:
    pd.DataFrame(trace).to_csv(RUN / "transcripts.csv", index=False, encoding="utf-8-sig")
    pd.DataFrame(exports, columns=["variant", "point", "summary", "checkpoint"]).to_csv(
        RUN / "summary_points.csv", index=False, encoding="utf-8-sig"
    )
    print("Ekspor tersimpan di", RUN.relative_to(ROOT))
    print("API lokal menerima satu teks dalam JSON melalui POST /summarize.")